# 03. 심화: Benchmark와 비용 tradeoff 분석

목표: OPD²가 기존 OPD보다 얼마나 이득을 주는지 toy benchmark table로 분석하고, teacher-base forward overhead를 함께 계산합니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. Toy benchmark 결과 구성

아래 수치는 논문과 AlphaXiv 요약의 대표 메시지를 반영한 학습용 표입니다. 실제 전체 benchmark 재현이 아니라, 여러 domain에서 평균을 내고 method를 비교하는 방법을 익히는 목적입니다.

In [ ]:
results = [
    {"model": "Qwen3-1.7B", "method": "baseline", "math": 34.8, "science": 40.2, "code": 28.5},
    {"model": "Qwen3-1.7B", "method": "OPD", "math": 51.0, "science": 45.5, "code": 35.6},
    {"model": "Qwen3-1.7B", "method": "ExOPD", "math": 51.4, "science": 46.0, "code": 36.0},
    {"model": "Qwen3-1.7B", "method": "OPD2", "math": 54.6, "science": 48.7, "code": 38.2},
    {"model": "Gemma4-E4B-it", "method": "baseline", "math": 60.6, "science": 52.0, "code": 43.0},
    {"model": "Gemma4-E4B-it", "method": "ExOPD", "math": 65.3, "science": 53.1, "code": 42.4},
    {"model": "Gemma4-E4B-it", "method": "OPD2", "math": 67.8, "science": 55.2, "code": 44.0},
]


def average_score(row):
    return (row["math"] + row["science"] + row["code"]) / 3


print("model | method | avg | math | science | code")
print("--- | --- | --- | --- | --- | ---")
for row in results:
    print(f"{row['model']} | {row['method']} | {average_score(row):.1f} | {row['math']:.1f} | {row['science']:.1f} | {row['code']:.1f}")

## 2. 모델별 최고 method 찾기

논문은 OPD²가 여러 모델 크기와 mode에서 일관되게 강하다고 주장합니다. 아래는 domain 평균 기준으로 가장 높은 method를 찾는 예입니다.

In [ ]:
models = sorted(set(row["model"] for row in results))
for model in models:
    rows = [row for row in results if row["model"] == model]
    best = max(rows, key=average_score)
    baseline = next(row for row in rows if row["method"] == "baseline")
    improvement = average_score(best) - average_score(baseline)
    print(f"{model}: best={best['method']}, avg={average_score(best):.1f}, improvement={improvement:.1f} points")

## 3. Overhead 대비 이득 계산

OPD²는 teacher-base forward가 추가되므로 학습 시간이 늘어납니다. 논문 요약은 대략 8-28% overhead를 언급합니다. 성능 이득이 충분하면 이 overhead는 감수할 수 있습니다.

In [ ]:
def training_time(base_hours, overhead_rate):
    return base_hours * (1 + overhead_rate)


base_hours = 10
overheads = [0.08, 0.18, 0.28]
print("overhead | total_hours | extra_hours")
print("--- | --- | ---")
for overhead in overheads:
    total = training_time(base_hours, overhead)
    print(f"{overhead:.0%} | {total:.1f} | {total - base_hours:.1f}")

## 4. Delta signal이 실패할 수 있는 경우 점검

Delta는 teacher와 teacher-base가 같은 학습 계보를 공유할 때 의미가 가장 선명합니다. 아래 checklist는 OPD² 적용 전 확인해야 할 조건입니다.

In [ ]:
checks = {
    "teacher_base_available": True,
    "same_tokenizer": True,
    "same_architecture_family": True,
    "reasoning_tuning_is_target_skill": True,
    "domain_eval_exists": True,
}


def readiness(checks):
    failed = [name for name, ok in checks.items() if not ok]
    if failed:
        return "not ready", failed
    return "ready", []


status, failed = readiness(checks)
print("status:", status)
print("failed checks:", failed)

checks_without_base = {**checks, "teacher_base_available": False}
status, failed = readiness(checks_without_base)
print("without teacher base ->", status, failed)